In [ ]:
import os
from importlib.util import find_spec
from IPython import get_ipython

IN_VSCODE = bool(os.environ.get('VSCODE_PID'))
if IN_VSCODE:
    PREFERRED_MATPLOTLIB_BACKENDS = [('widget', 'module://ipympl.backend_nbagg'), ('qt', 'qt'), ('notebook', 'notebook')]
else:
    PREFERRED_MATPLOTLIB_BACKENDS = [('widget', 'module://ipympl.backend_nbagg'), ('notebook', 'notebook'), ('qt', 'qt')]
ip = get_ipython()
if IN_VSCODE:
    print('VS Code detected; trying %matplotlib widget first, then falling back to qt/notebook if needed')

for MATPLOTLIB_BACKEND, MATPLOTLIB_MAGIC_ARG in PREFERRED_MATPLOTLIB_BACKENDS:
    if MATPLOTLIB_BACKEND == 'widget' and find_spec('ipympl') is None:
        print("ipympl is not installed; skipping %matplotlib widget")
        continue
    try:
        if MATPLOTLIB_BACKEND == 'widget':
            import ipympl
        ip.run_line_magic('matplotlib', MATPLOTLIB_MAGIC_ARG)
        print(f"Using %matplotlib {MATPLOTLIB_BACKEND}")
        break
    except Exception as exc:
        print(f"Failed to enable %matplotlib {MATPLOTLIB_BACKEND}: {exc}")
else:
    MATPLOTLIB_BACKEND = 'inline'
    ip.run_line_magic('matplotlib', MATPLOTLIB_BACKEND)
    print("Falling back to %matplotlib inline")

from pathlib import Path
from data_paths import AW_DIR, BIOEXP
import importlib
import numpy as np
import phyto_sibciom

importlib.reload(phyto_sibciom)

from phyto_sibciom import inspect_nc, make_file_path, plot_field_from_click, prepare_sibciom_3dvar


In [ ]:
DATA_DIR = BIOEXP / 'npzd_20_03_2026' / 'ocn'

VAR_NAME = 'sal'
SNAP_YEAR = 2012
SNAP_MONTH = 8
LEVEL = 0
PROFILE_MAX_DEPTH_M = 200.0
INITIAL_I = 191
INITIAL_J = 218
NC_INFO = False


In [ ]:
selected_file = make_file_path(DATA_DIR, SNAP_YEAR, SNAP_MONTH)
print(f'Selected file: {selected_file}')

if NC_INFO:
    inspect_nc(selected_file)

prepared = prepare_sibciom_3dvar(
    data_dir=DATA_DIR,
    year=SNAP_YEAR,
    month=SNAP_MONTH,
    var_name=VAR_NAME,
    grid_base_dir=AW_DIR,
)

i, j = plot_field_from_click(
    var=np.asarray(prepared['var_3d']),
    z_levels=np.asarray(prepared['levels']),
    year=SNAP_YEAR,
    month=SNAP_MONTH,
    map_level=LEVEL,
    profile_max_depth=PROFILE_MAX_DEPTH_M,
    i=INITIAL_I,
    j=INITIAL_J,
    var_name=VAR_NAME,
    var_units=str(prepared['units']),
)

print(f'Selected point: i={i}, j={j}')
